# Food-101 Colab GPU Training Workflow

This notebook uses Colab as temporary GPU compute. GitHub provides the repository source, `/content` stores temporary data and training outputs, and Google Drive is used only when explicitly archiving selected final runs.

The final report remains `food101_CNN_final_project.ipynb`.

## 1. Runtime Setup

Use `Runtime -> Change runtime type -> GPU` in Colab. The setup cell clones the configured GitHub branch into `/content/food101-cnn`, installs the package, and creates temporary runtime folders.

In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
GITHUB_REPO_URL = "https://github.com/alexal00/food101-cnn.git"
GITHUB_BRANCH = "main"  # Change to codex/final-deliverable-cleanup while testing this branch.
CLONE_FRESH = True
RUN_INSTALL = True

CONTENT_ROOT = Path("/content") if IN_COLAB else Path.cwd().resolve()
PROJECT_ROOT = CONTENT_ROOT / "food101-cnn" if IN_COLAB else Path.cwd().resolve()
RUN_ROOT = CONTENT_ROOT / "food101-runs" if IN_COLAB else PROJECT_ROOT / "outputs" / "runs"
DRIVE_FINAL_RUN_ROOT = Path("/content/drive/MyDrive/food101-cnn/final-runs")


def run_setup_command(command: list[str], *, cwd: Path | None = None) -> None:
    print("$", " ".join(command))
    subprocess.run(command, cwd=cwd, check=True)


if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    if PROJECT_ROOT.exists() and CLONE_FRESH:
        shutil.rmtree(PROJECT_ROOT)
    if not PROJECT_ROOT.exists():
        run_setup_command([
            "git",
            "clone",
            "--depth",
            "1",
            "--branch",
            GITHUB_BRANCH,
            GITHUB_REPO_URL,
            str(PROJECT_ROOT),
        ])
else:
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / "pyproject.toml").is_file():
            PROJECT_ROOT = candidate
            RUN_ROOT = PROJECT_ROOT / "outputs" / "runs"
            break

os.chdir(PROJECT_ROOT)
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

pythonpath_parts = [str(SRC_ROOT), *(part for part in os.environ.get("PYTHONPATH", "").split(os.pathsep) if part)]
os.environ["PYTHONPATH"] = os.pathsep.join(dict.fromkeys(pythonpath_parts))

for directory in [PROJECT_ROOT / "data" / "raw", PROJECT_ROOT / "data" / "processed", PROJECT_ROOT / "data" / "reports", RUN_ROOT]:
    directory.mkdir(parents=True, exist_ok=True)
if IN_COLAB:
    DRIVE_FINAL_RUN_ROOT.mkdir(parents=True, exist_ok=True)

if RUN_INSTALL:
    run_setup_command([sys.executable, "-m", "pip", "install", "-e", ".[dev]"], cwd=PROJECT_ROOT)

required = [
    "pyproject.toml",
    "src/food101_cnn/config.py",
    "scripts/run_experiment_plan.py",
    "scripts/train.py",
    "scripts/evaluate.py",
    "scripts/export_model.py",
    "configs/baseline_cnn_simple.yaml",
    "configs/efficientnet_b0_gpu.yaml",
]
missing = [item for item in required if not (PROJECT_ROOT / item).exists()]
if missing:
    raise FileNotFoundError(f"Repository checkout is missing required files: {missing}")

print(json.dumps({
    "in_colab": IN_COLAB,
    "github_repo": GITHUB_REPO_URL,
    "github_branch": GITHUB_BRANCH,
    "project_root": str(PROJECT_ROOT),
    "run_root": str(RUN_ROOT),
    "drive_final_run_root": str(DRIVE_FINAL_RUN_ROOT) if IN_COLAB else None,
    "pythonpath_first": os.environ["PYTHONPATH"].split(os.pathsep)[0],
}, indent=2))

## 2. Environment Check

This verifies the installed package and available accelerator before any long-running command is launched.

In [ ]:
import torch
import food101_cnn


def gpu_info() -> dict:
    if not torch.cuda.is_available():
        return {"device": "cpu", "name": "CPU", "memory_gb": 0.0}
    props = torch.cuda.get_device_properties(0)
    return {
        "device": "cuda",
        "name": props.name,
        "memory_gb": round(props.total_memory / 1024**3, 2),
    }


GPU = gpu_info()
print(json.dumps({
    "food101_cnn_version": food101_cnn.__version__,
    "module_path": food101_cnn.__file__,
    "gpu": GPU,
}, indent=2))

## 3. Execution Flags

Default values print the selected plan instead of launching expensive jobs. Data and run products stay in the runtime filesystem unless final archiving is enabled.

In [ ]:
RUN_DATA_PREP = False
RUN_IMAGE_CACHE = False
RUN_TRAINING = False
RUN_EVALUATION = False
RUN_EXPORT_PT = False
RUN_ARCHIVE_FINAL_RUNS = False

PLAN_NAME = "gpu-default"  # Use "gpu-full" for the full comparison set.
PLAN_MODELS: list[str] = []  # Optional subset, for example ["efficientnet_b0"].
DATA_CONFIG = "configs/efficientnet_b0.yaml"

INDEX_CSV = PROJECT_ROOT / "data" / "reports" / "dataset_index.csv"
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed"

RESUME_CHECKPOINTS = {
    # "resnet50": "/content/food101-runs/<run>/checkpoints/<run>_best_model.pt",
}

FINAL_RUN_MODELS = [
    # "efficientnet_b0",
]

print(json.dumps({
    "plan": PLAN_NAME,
    "models": PLAN_MODELS or "all plan models",
    "index_csv": str(INDEX_CSV),
    "processed_root": str(PROCESSED_ROOT),
    "run_root": str(RUN_ROOT),
    "archive_enabled": RUN_ARCHIVE_FINAL_RUNS,
}, indent=2))

## 4. Plan Runner

The training plan is executed through `scripts/run_experiment_plan.py`, which delegates to `scripts/download_data.py`, `scripts/cache_images.py`, `scripts/train.py`, `scripts/evaluate.py`, and `scripts/export_model.py`.

In [ ]:
from food101_cnn.utils.runs import latest_run_for_model, list_run_manifests, manifest_to_comparison_row


def run_cli(command: list[str]) -> subprocess.CompletedProcess:
    print("$", " ".join(str(item) for item in command))
    process = subprocess.Popen(
        [str(item) for item in command],
        cwd=PROJECT_ROOT,
        env=os.environ.copy(),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    output_tail: list[str] = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        output_tail.append(line)
        output_tail = output_tail[-80:]
    returncode = process.wait()
    if returncode != 0:
        raise RuntimeError(
            f"Command failed with exit code {returncode}: {' '.join(str(item) for item in command)}\n"
            f"Last command output:\n{''.join(output_tail)}"
        )
    return subprocess.CompletedProcess(command, returncode)


def build_plan_command() -> list[str]:
    command = [
        sys.executable,
        "scripts/run_experiment_plan.py",
        "--plan",
        PLAN_NAME,
        "--data-config",
        DATA_CONFIG,
        "--index-csv",
        str(INDEX_CSV),
        "--cache-root",
        str(PROCESSED_ROOT),
        "--run-root",
        str(RUN_ROOT),
        "--gpu-memory-gb",
        str(GPU["memory_gb"]),
        "--device",
        "cuda" if torch.cuda.is_available() else "cpu",
    ]
    for model_name in PLAN_MODELS:
        command.extend(["--model", model_name])
    for model_name, checkpoint in RESUME_CHECKPOINTS.items():
        command.extend(["--resume-checkpoint", f"{model_name}={checkpoint}"])
    if RUN_DATA_PREP:
        command.append("--data-prep")
    if RUN_IMAGE_CACHE:
        command.append("--cache-images")
    if RUN_TRAINING:
        command.append("--train")
    if RUN_EVALUATION:
        command.append("--evaluate")
    if RUN_EXPORT_PT:
        command.append("--export-pt")
    if not any([RUN_DATA_PREP, RUN_IMAGE_CACHE, RUN_TRAINING, RUN_EVALUATION, RUN_EXPORT_PT]):
        command.append("--dry-run")
    return command


## 5. Execute Plan

With all execution flags set to `False`, this cell prints the selected plan. Enable one or more flags above to run data preparation, caching, training, evaluation, or export.

In [ ]:
plan_command = build_plan_command()
run_cli(plan_command)

## 6. Final Run Archive

Only complete selected runs are copied to Drive. A run without `manifest.json` is treated as partial and is not archived.

In [ ]:
def archive_complete_run(model_name: str) -> Path | None:
    if not IN_COLAB:
        print("Drive archiving is only available in Colab.")
        return None
    manifest = latest_run_for_model(PROJECT_ROOT, model_name, run_root=RUN_ROOT)
    if not manifest:
        print(f"No complete run manifest found for {model_name}")
        return None
    run_name = manifest["run_name"]
    source = RUN_ROOT / run_name
    manifest_path = source / "manifest.json"
    if not manifest_path.is_file():
        print(f"Skipping partial run without manifest: {source}")
        return None
    destination = DRIVE_FINAL_RUN_ROOT / run_name
    shutil.copytree(source, destination, dirs_exist_ok=True)
    print(f"archived {source} -> {destination}")
    return destination


archived = []
if RUN_ARCHIVE_FINAL_RUNS:
    for model_name in FINAL_RUN_MODELS:
        archived_path = archive_complete_run(model_name)
        if archived_path is not None:
            archived.append(str(archived_path))
else:
    print("Final-run archiving disabled.")

archived

## 7. Run Manifest Snapshot

Use this quick table to confirm what the runtime produced. The final report notebook performs the project-level comparison.

In [ ]:
import pandas as pd

manifests = list_run_manifests(PROJECT_ROOT, run_root=RUN_ROOT)
rows = [manifest_to_comparison_row(item) for item in manifests]
pd.DataFrame(rows, columns=["model", "version", "trained_at", "checkpoint", "top1", "top5", "macro_f1", "ece", "latency_ms", "params", "notes"])